## Data Loading

In [5]:
# Proof-of-concept: scrape one confederation WCQ stats page directly.
import pandas as pd

URL = "https://fbref.com/en/comps/6/stats/WCQ----UEFA-M-Stats"
CACHE = CACHE_DIR / "wcq_uefa_2026.html"

reader = fbref.get(URL, CACHE)
tree = html.parse(reader)
parser = etree.HTMLParser(recover=True)

# The main player stats table on a comp's /stats/ page is usually `stats_standard`
# (no comp-id suffix because the page IS that comp). FBref often comments it out.
candidates = []
for c in tree.xpath("//comment()[contains(., 'stats_standard')]"):
    root = etree.fromstring(f"<root>{c.text}</root>", parser)
    candidates.extend(root.xpath(".//table[contains(@id, 'stats_standard')]"))
candidates.extend(tree.xpath("//table[contains(@id, 'stats_standard')]"))

# Dedupe by id
seen = set()
unique = []
for t in candidates:
    if t.get("id") in seen:
        continue
    seen.add(t.get("id"))
    unique.append(t)

print(f"Found {len(unique)} stats_standard table(s): {[t.get('id') for t in unique]}")

# Parse the first one
df = _parse_table(unique[0])
print(f"\nShape: {df.shape}")
print(f"\nTop-level column groups: {df.columns.get_level_values(0).unique().tolist()}")
print(f"\nLeaf column names: {df.columns.get_level_values(1).tolist()}")

# Inspect — what does the Squad/Nation column look like?
print(f"\nFirst 3 rows:")
print(df.head(3))

# Flatten so we can poke at it
df_flat = df.copy()
df_flat.columns = [b if (not a or a == b or a.startswith("Unnamed")) else f"{a}_{b}" for a, b in df_flat.columns]
df_flat = df_flat.reset_index(drop=True)

# Find the team-affiliation column
team_cols = [c for c in df_flat.columns if c.lower() in ("squad", "team", "nation")]
print(f"\nTeam-affiliation columns: {team_cols}")

if team_cols:
    tc = team_cols[0]
    print(f"\nUnique {tc} values ({df_flat[tc].nunique()}):")
    print(sorted(df_flat[tc].dropna().unique().tolist()))

print(f"\nTotal players: {len(df_flat)}")

Found 1 stats_standard table(s): ['stats_standard']

Shape: (1590, 24)

Top-level column groups: ['Unnamed: 0_level_0', 'Unnamed: 1_level_0', 'Unnamed: 2_level_0', 'Unnamed: 3_level_0', 'Unnamed: 4_level_0', 'Unnamed: 5_level_0', 'Playing Time', 'Performance', 'Per 90 Minutes', 'Unnamed: 23_level_0']

Leaf column names: ['Rk', 'Player', 'Pos', 'Squad', 'Age', 'Born', 'MP', 'Starts', 'Min', '90s', 'Gls', 'Ast', 'G+A', 'G-PK', 'PK', 'PKatt', 'CrdY', 'CrdR', 'Gls', 'Ast', 'G+A', 'G-PK', 'G+A-PK', 'Matches']

First 3 rows:
  Unnamed: 0_level_0  Unnamed: 1_level_0 Unnamed: 2_level_0  \
                  Rk              Player                Pos   
0                  1  Bárður Á Reynatrøð                 GK   
1                  2      Thelo Aasgaard                 MF   
2                  3          Liel Abada                 DF   

  Unnamed: 3_level_0 Unnamed: 4_level_0 Unnamed: 5_level_0 Playing Time  \
               Squad                Age               Born           MP   
0      Fa

In [6]:
# Find URLs for all other "FIFA World Cup Qualification" comps from FBref's master comps index.
leagues_html = Path.home() / "soccerdata" / "data" / "FBref" / "leagues.html"
tree = html.parse(str(leagues_html))

# Find all anchor tags with text matching "World Cup Qualification"
wcq_links = []
for a in tree.xpath("//a"):
    text = (a.text or "").strip()
    href = a.get("href", "")
    if "World Cup Qualification" in text and "Women" not in text:
        wcq_links.append((text, href))

print(f"Found {len(wcq_links)} WCQ links:")
for text, href in wcq_links:
    print(f"  {text:55s} → {href}")


Found 7 WCQ links:
  FIFA World Cup Qualification — Inter-confederation play-offs → /en/comps/255/history/FIFA-World-Cup-Qualification----Inter-confederation-play-offs-Seasons
  FIFA World Cup Qualification — CAF                      → /en/comps/2/history/WCQ----CAF-M-Seasons
  FIFA World Cup Qualification — CONCACAF                 → /en/comps/3/history/WCQ----CONCACAF-M-Seasons
  FIFA World Cup Qualification — CONMEBOL                 → /en/comps/4/history/WCQ----CONMEBOL-M-Seasons
  FIFA World Cup Qualification — OFC                      → /en/comps/5/history/WCQ----OFC-M-Seasons
  FIFA World Cup Qualification — UEFA                     → /en/comps/6/history/WCQ----UEFA-M-Seasons
  FIFA World Cup Qualification — AFC                      → /en/comps/7/history/WCQ----AFC-M-Seasons


In [7]:
# Scrape all 7 WCQ pages → one combined player-stats DataFrame for the 2026 cycle.
import re

CONFED_URLS = {
    "UEFA":      "https://fbref.com/en/comps/6/stats/WCQ----UEFA-M-Stats",
    "CAF":       "https://fbref.com/en/comps/2/stats/WCQ----CAF-M-Stats",
    "CONCACAF":  "https://fbref.com/en/comps/3/stats/WCQ----CONCACAF-M-Stats",
    "CONMEBOL":  "https://fbref.com/en/comps/4/stats/WCQ----CONMEBOL-M-Stats",
    "OFC":       "https://fbref.com/en/comps/5/stats/WCQ----OFC-M-Stats",
    "AFC":       "https://fbref.com/en/comps/7/stats/WCQ----AFC-M-Stats",
    "InterConf": "https://fbref.com/en/comps/255/stats/FIFA-World-Cup-Qualification----Inter-confederation-play-offs-Stats",
}

def fetch_and_parse_wcq(confed, url):
    cache = CACHE_DIR / f"wcq_{confed}_2026.html"
    reader = fbref.get(url, cache)
    tree = html.parse(reader)
    parser = etree.HTMLParser(recover=True)

    candidates = []
    for c in tree.xpath("//comment()[contains(., 'stats_standard')]"):
        root = etree.fromstring(f"<root>{c.text}</root>", parser)
        candidates.extend(root.xpath(".//table[contains(@id, 'stats_standard')]"))
    candidates.extend(tree.xpath("//table[contains(@id, 'stats_standard')]"))

    seen, unique = set(), []
    for t in candidates:
        if t.get("id") in seen:
            continue
        seen.add(t.get("id"))
        unique.append(t)
    if not unique:
        return None
    df = _parse_table(unique[0])
    # Flatten MultiIndex columns
    df.columns = [b if (not a or a == b or str(a).startswith("Unnamed")) else f"{a}_{b}"
                  for a, b in df.columns]
    df = df.reset_index(drop=True)
    df["confederation"] = confed
    return df

frames = []
for confed, url in CONFED_URLS.items():
    df = fetch_and_parse_wcq(confed, url)
    if df is None:
        print(f"  {confed}: NO TABLE FOUND")
        continue
    print(f"  {confed}: {len(df)} player rows, {df['Squad'].nunique()} squads")
    frames.append(df)

wcq_all = pd.concat(frames, ignore_index=True)
print(f"\n=== Combined: {len(wcq_all)} player rows across {wcq_all['Squad'].nunique()} nations ===")
print(f"Columns: {wcq_all.columns.tolist()}")

# Cross-check coverage against our WC squads
wc_players = pd.read_csv("../data/processed/player_fixtures.csv")[["player", "team"]].drop_duplicates()
wc_nations = sorted(wc_players["team"].unique())
print(f"\nWC nations in our roster: {len(wc_nations)}")

wcq_nations = set(wcq_all["Squad"].dropna().unique())
present     = [n for n in wc_nations if n in wcq_nations]
missing     = [n for n in wc_nations if n not in wcq_nations]
print(f"Present in WCQ scrape: {len(present)} / {len(wc_nations)}")
print(f"Missing (likely name-mismatch — fix with a mapping): {missing}")

  UEFA: 1590 player rows, 54 squads
  CAF: 2100 player rows, 53 squads
  CONCACAF: 952 player rows, 32 squads
  CONMEBOL: 462 player rows, 10 squads
  OFC: 227 player rows, 11 squads
  AFC: 1517 player rows, 46 squads
  InterConf: NO TABLE FOUND

=== Combined: 6848 player rows across 206 nations ===
Columns: ['Rk', 'Player', 'Pos', 'Squad', 'Age', 'Born', 'Playing Time_MP', 'Playing Time_Starts', 'Playing Time_Min', 'Playing Time_90s', 'Performance_Gls', 'Performance_Ast', 'Performance_G+A', 'Performance_G-PK', 'Performance_PK', 'Performance_PKatt', 'Performance_CrdY', 'Performance_CrdR', 'Per 90 Minutes_Gls', 'Per 90 Minutes_Ast', 'Per 90 Minutes_G+A', 'Per 90 Minutes_G-PK', 'Per 90 Minutes_G+A-PK', 'Matches', 'confederation']

WC nations in our roster: 48
Present in WCQ scrape: 43 / 48
Missing (likely name-mismatch — fix with a mapping): ['Bosnia and Herzegovina', 'Cabo Verde', 'Canada', 'Mexico', 'USA']


In [58]:
# Verify Cape Verde / Cabo Verde mapping, then build the filtered international-stats table.

# Sanity: list all FBref Squad names that look like Cape Verde
print("Squad names containing 'verde' or 'cape':")
for s in sorted(wcq_all["Squad"].dropna().unique()):
    if "verde" in s.lower() or "cape" in s.lower():
        print(f"  {s}")

# Map our WC squad names → FBref Squad names. Only entries that actually differ.
WC_TO_FBREF_SQUAD = {
    "Bosnia and Herzegovina": "Bosnia-Herzegovina",
    "Cabo Verde": "Cape Verde",
    # Add more here if other mismatches surface
}

# Apply mapping to our roster, filter wcq_all to our 48
wc_players["fbref_squad"] = wc_players["team"].map(WC_TO_FBREF_SQUAD).fillna(wc_players["team"])
wc_squads_fbref = set(wc_players["fbref_squad"].unique())

intl = wcq_all[wcq_all["Squad"].isin(wc_squads_fbref)].copy()

print(f"\nFiltered international-stats rows: {len(intl)}")
print(f"Squads represented: {intl['Squad'].nunique()} / 48")

# Per-team match/minute summary
summary = (
    intl.assign(min_=pd.to_numeric(intl["Playing Time_Min"].astype(str).str.replace(",", ""), errors="coerce"))
        .groupby("Squad")
        .agg(players=("Player", "nunique"),
             total_min=("min_", "sum"),
             max_mp=("Playing Time_MP", "max"))
        .sort_values("max_mp", ascending=False)
)
print(f"\nPer-team summary (top 10 by matches played):")
print(summary.head(10))
print(f"\nPer-team summary (bottom 10 by matches played):")
print(summary.tail(10))

# Which of the 48 still have zero rows (should be USA, Canada, Mexico)
covered = set(intl["Squad"].unique())
still_missing = [t for t in wc_squads_fbref if t not in covered]
print(f"\nStill missing (expected to be the 3 co-hosts): {still_missing}")

Squad names containing 'verde' or 'cape':
  Cape Verde

Filtered international-stats rows: 1653
Squads represented: 45 / 48

Per-team summary (top 10 by matches played):
                players  total_min  max_mp
Squad                                     
Iraq                 48      19650      20
Ecuador              44      17706      18
Brazil               60      17810      18
Colombia             43      17816      18
Argentina            35      17738      18
Uruguay              41      17820      17
Paraguay             46      17773      17
Qatar                56      17782      16
Korea Republic       52      15840      16
Jordan               37      15840      16

Per-team summary (bottom 10 by matches played):
             players  total_min  max_mp
Squad                                  
Austria           28       7919       8
Croatia           32       7920       8
England           32       7920       8
Switzerland       23       5940       6
France            30     

In [59]:
out = "../data/processed/international_wcq_2026.csv"
intl.to_csv(out, index=False)
print(f"Wrote {len(intl)} rows → {out}")

Wrote 1653 rows → ../data/processed/international_wcq_2026.csv


## More Data!! 

We want to get more international football player level data, e.g. Copa America for South American nations, UEFA Nations League for UEFA nations....

In [10]:
# Discover URLs for the 5 v2 comps from FBref's master comps index.
search_terms = [
    "UEFA Nations League",
    "Gold Cup",
    "Africa Cup of Nations",
    "UEFA Euro",
    "Copa Am",
]

leagues_html = Path.home() / "soccerdata" / "data" / "FBref" / "leagues.html"
tree = html.parse(str(leagues_html))

print("Candidate links per search term:\n")
for term in search_terms:
    print(f"=== {term!r} ===")
    matches = []
    for a in tree.xpath("//a"):
        text = (a.text or "").strip()
        href = a.get("href", "")
        if term.lower() in text.lower() and "Women" not in text and "U-" not in text and "Youth" not in text:
            matches.append((text, href))
    if not matches:
        print("  (no matches)")
    for text, href in matches:
        print(f"  {text:55s} → {href}")
    print()


Candidate links per search term:

=== 'UEFA Nations League' ===
  UEFA Nations League                                     → /en/comps/677/UEFA-Nations-League-Stats
  UEFA Nations League                                     → /en/comps/677/history/UEFA-Nations-League-Seasons
  UEFA Nations League                                     → /en/comps/677/UEFA-Nations-League-Stats

=== 'Gold Cup' ===
  CONCACAF Gold Cup                                       → /en/comps/681/history/Gold-Cup-Seasons
  CONCACAF Gold Cup                                       → /en/comps/681/Gold-Cup-Stats

=== 'Africa Cup of Nations' ===
  Africa Cup of Nations                                   → /en/comps/656/history/Africa-Cup-of-Nations-Seasons
  Africa Cup of Nations qualification                     → /en/comps/657/history/Africa-Cup-of-Nations-qualification-Seasons
  Africa Cup of Nations                                   → /en/comps/656/Africa-Cup-of-Nations-Stats

=== 'UEFA Euro' ===
  UEFA European Football

In [ ]:
# for _, comp, _ in V2_COMPS:
#     cache_file = CACHE_DIR / f"{comp.replace(' ', '_')}.html"
#     if cache_file.exists():
#         cache_file.unlink()
#         print(f"Deleted {cache_file.name}")

Deleted UEFA_Nations_League.html
Deleted AFCON_2025.html
Deleted Gold_Cup_2025.html
Deleted Euro_2024.html
Deleted Copa_America_2024.html


In [15]:
# v2: add 5 more international competitions, with a `competition` column for downstream filtering.

V2_COMPS = [
    ("UEFA",     "UEFA Nations League",   "https://fbref.com/en/comps/677/stats/UEFA-Nations-League-Stats"),
    ("CAF",      "AFCON 2025",            "https://fbref.com/en/comps/656/stats/Africa-Cup-of-Nations-Stats"),
    ("CONCACAF", "Gold Cup 2025",         "https://fbref.com/en/comps/681/stats/Gold-Cup-Stats"),
    ("UEFA",     "Euro 2024",             "https://fbref.com/en/comps/676/stats/UEFA-Euro-Stats"),
    ("CONMEBOL", "Copa America 2024",     "https://fbref.com/en/comps/685/stats/Copa-America-Stats"),
]

def fetch_comp_standard(url, cache_name):
    cache = CACHE_DIR / cache_name
    reader = fbref.get(url, cache)
    tree = html.parse(reader)
    parser = etree.HTMLParser(recover=True)

    candidates = []
    for c in tree.xpath("//comment()[contains(., 'stats_standard')]"):
        root = etree.fromstring(f"<root>{c.text}</root>", parser)
        candidates.extend(root.xpath(".//table[contains(@id, 'stats_standard')]"))
    candidates.extend(tree.xpath("//table[contains(@id, 'stats_standard')]"))
    seen, unique = set(), []
    for t in candidates:
        if t.get("id") in seen:
            continue
        seen.add(t.get("id"))
        unique.append(t)
    if not unique:
        return None, None

    tbl = unique[0]
    caption_el = tbl.xpath(".//caption")
    caption = "".join(caption_el[0].itertext()).strip() if caption_el else tbl.get("id", "")

    df = _parse_table(tbl)
    df.columns = [b if (not a or a == b or str(a).startswith("Unnamed")) else f"{a}_{b}"
                  for a, b in df.columns]
    df = df.reset_index(drop=True)
    return caption, df

v2_frames = []
for confed, comp, url in V2_COMPS:
    cache_name = f"{comp.replace(' ', '_')}.html"
    caption, df = fetch_comp_standard(url, cache_name)
    if df is None:
        print(f"  {comp}: NO TABLE FOUND ({url})")
        continue
    df["confederation"] = confed
    df["competition"] = comp
    print(f"  {comp:25s} | caption: {caption!r}")
    print(f"  {'':25s}   {len(df)} player rows, {df['Squad'].nunique()} squads")
    v2_frames.append(df)

v2_all = pd.concat(v2_frames, ignore_index=True)

# Add `competition` column to v1 (intl) so schemas align
intl["competition"] = intl["confederation"].map({
    "UEFA": "UEFA WCQ", "CAF": "CAF WCQ", "CONCACAF": "CONCACAF WCQ",
    "CONMEBOL": "CONMEBOL WCQ", "OFC": "OFC WCQ", "AFC": "AFC WCQ",
})

# Filter v2 to our 48 WC squads (reuse the mapping from v1)
v2_filtered = v2_all[v2_all["Squad"].isin(wc_squads_fbref)].copy()
print(f"\nv2 filtered to WC squads: {len(v2_filtered)} rows across {v2_filtered['Squad'].nunique()} squads")

# Stack v1 + v2
intl_all = pd.concat([intl, v2_filtered], ignore_index=True)
print(f"\n=== Combined v1+v2: {len(intl_all)} rows, {intl_all['Squad'].nunique()} squads ===")
print(intl_all.groupby("competition").size().sort_values(ascending=False))

# Co-host check — did we pick up USA/Canada/Mexico?
cohost_data = intl_all[intl_all["Squad"].isin(["United States", "USA", "Canada", "Mexico"])]
print(f"\nCo-host coverage now:")
print(cohost_data.groupby(["Squad", "competition"]).size())

  UEFA Nations League       | caption: 'Player Standard Stats 2024-2025 UEFA Nations League Table'
                              1457 player rows, 54 squads
  AFCON 2025                | caption: 'Player Standard Stats 2025 Africa Cup of Nations Table'
                              543 player rows, 24 squads
  Gold Cup 2025             | caption: 'Player Standard Stats 2025 Gold Cup Table'
                              335 player rows, 16 squads
  Euro 2024                 | caption: 'Player Standard Stats 2024 UEFA Euro 2024 Table'
                              493 player rows, 24 squads
  Copa America 2024         | caption: 'Player Standard Stats 2024 Copa América Table'
                              339 player rows, 16 squads

v2 filtered to WC squads: 1248 rows across 36 squads

=== Combined v1+v2: 2901 rows, 47 squads ===
competition
UEFA Nations League    480
UEFA WCQ               470
AFC WCQ                409
CAF WCQ                382
Euro 2024              270
CONMEBOL WCQ 

In [60]:
# Check FBref's spelling for USA
print("Squad values containing 'unit' or 'state':")
for s in sorted(v2_all["Squad"].dropna().unique()):
    if "unit" in s.lower() or "state" in s.lower():
        print(f"  {s}")

# Add USA to the mapping and re-filter
WC_TO_FBREF_SQUAD["USA"] = "United States"

wc_players["fbref_squad"] = wc_players["team"].map(WC_TO_FBREF_SQUAD).fillna(wc_players["team"])
wc_squads_fbref = set(wc_players["fbref_squad"].unique())

# Re-filter both v1 and v2 with the updated mapping
intl_v1 = wcq_all[wcq_all["Squad"].isin(wc_squads_fbref)].copy()
intl_v1["competition"] = intl_v1["confederation"].map({
    "UEFA": "UEFA WCQ", "CAF": "CAF WCQ", "CONCACAF": "CONCACAF WCQ",
    "CONMEBOL": "CONMEBOL WCQ", "OFC": "OFC WCQ", "AFC": "AFC WCQ",
})
intl_v2 = v2_all[v2_all["Squad"].isin(wc_squads_fbref)].copy()

intl_all = pd.concat([intl_v1, intl_v2], ignore_index=True)
print(f"\nCombined v1+v2: {len(intl_all)} rows, {intl_all['Squad'].nunique()} squads")

# Co-host coverage now
cohost_data = intl_all[intl_all["Squad"].isin(["United States", "Canada", "Mexico"])]
print("\nCo-host coverage:")
print(cohost_data.groupby(["Squad", "competition"]).size())

# Save final v1+v2 international stats
out = "../data/processed/international_stats_2026.csv"
intl_all.to_csv(out, index=False)
print(f"\nWrote {len(intl_all)} rows → {out}")


Squad values containing 'unit' or 'state':
  United States

Combined v1+v2: 2945 rows, 48 squads

Co-host coverage:
Squad          competition      
Canada         Copa America 2024    22
               Gold Cup 2025        23
Mexico         Copa America 2024    20
               Gold Cup 2025        23
United States  Copa America 2024    21
               Gold Cup 2025        23
dtype: int64

Wrote 2945 rows → ../data/processed/international_stats_2026.csv


## Calculating Average Opponent Elo Strength

In [61]:
elo = pd.read_csv("../data/elo_ratings.csv")  # adjust path if different
print(elo.shape)
print(elo.columns.tolist())
print(elo.head(10))
print(f"\nUnique teams: {elo['team'].nunique() if 'team' in elo.columns else 'check column name'}")

(211, 92)
['Rank', 'Code', 'Country', 'PELE', '△ 1 year', '_2005Q0', '_2005Q1', '_2005Q2', '_2005Q3', '_2005Q4', '_2006Q1', '_2006Q2', '_2006Q3', '_2006Q4', '_2007Q1', '_2007Q2', '_2007Q3', '_2007Q4', '_2008Q1', '_2008Q2', '_2008Q3', '_2008Q4', '_2009Q1', '_2009Q2', '_2009Q3', '_2009Q4', '_2010Q1', '_2010Q2', '_2010Q3', '_2010Q4', '_2011Q1', '_2011Q2', '_2011Q3', '_2011Q4', '_2012Q1', '_2012Q2', '_2012Q3', '_2012Q4', '_2013Q1', '_2013Q2', '_2013Q3', '_2013Q4', '_2014Q1', '_2014Q2', '_2014Q3', '_2014Q4', '_2015Q1', '_2015Q2', '_2015Q3', '_2015Q4', '_2016Q1', '_2016Q2', '_2016Q3', '_2016Q4', '_2017Q1', '_2017Q2', '_2017Q3', '_2017Q4', '_2018Q1', '_2018Q2', '_2018Q3', '_2018Q4', '_2019Q1', '_2019Q2', '_2019Q3', '_2019Q4', '_2020Q1', '_2020Q2', '_2020Q3', '_2020Q4', '_2021Q1', '_2021Q2', '_2021Q3', '_2021Q4', '_2022Q1', '_2022Q2', '_2022Q3', '_2022Q4', '_2023Q1', '_2023Q2', '_2023Q3', '_2023Q4', '_2024Q1', '_2024Q2', '_2024Q3', '_2024Q4', '_2025Q1', '_2025Q2', '_2025Q3', '_2025Q4', '_2026Q

In [22]:
import re

# === Build clean ELO lookup ===
elo = pd.read_csv("../data/elo_ratings.csv")
elo["country_clean"] = elo["Country"].apply(
    lambda s: re.sub(r":[a-z\-]+:\s*", "", str(s)).replace("🏆", "").strip()
)
print("Sample ELO country names:", elo["country_clean"].head(20).tolist())

# === Map FBref Squad names → ELO country names ===
# We need this for both wcq_all + v2_all (which together represent all comp participants).
FBREF_TO_ELO = {
    # Diacritic/punctuation differences
    "Bosnia-Herzegovina":   "Bosnia/Herzegovina",
    "Côte d'Ivoire":        "Cote d'Ivoire",
    "Curaçao":              "Curacao",
    # Abbreviations
    "Congo DR":             "Dem. Rep. Congo",
    "Congo":                "Rep, Congo",
    "Rep. of Ireland":      "Rep. Ireland",
    "N. Macedonia":         "North Macedonia",
    "UAE":                  "Unit. Arab Emir.",
    "China PR":             "China",
    "CAR":                  "Cent. Afr. Rep.",
    # Already-working entries (keep explicit for clarity)
    "Korea Republic":       "South Korea",
    "United States":        "United States",
    "IR Iran":              "Iran",
}

intl_full = pd.concat([wcq_all, v2_all], ignore_index=True)
intl_full["competition"] = intl_full["competition"].fillna(
    intl_full["confederation"].map({
        "UEFA": "UEFA WCQ", "CAF": "CAF WCQ", "CONCACAF": "CONCACAF WCQ",
        "CONMEBOL": "CONMEBOL WCQ", "OFC": "OFC WCQ", "AFC": "AFC WCQ",
    })
)
intl_full["elo_country"] = intl_full["Squad"].map(FBREF_TO_ELO).fillna(intl_full["Squad"])

# === Compute mean PELE per competition (over participating squads) ===
elo_lookup = elo.set_index("country_clean")["PELE"].to_dict()

participants = intl_full.groupby("competition")["elo_country"].unique().to_dict()
rows = []
unmatched = set()
for comp, squads in participants.items():
    pele_vals = []
    for s in squads:
        if s in elo_lookup:
            pele_vals.append(elo_lookup[s])
        else:
            unmatched.add(s)
    rows.append({"competition": comp, "n_squads": len(squads), "n_matched": len(pele_vals),
                 "mean_pele": sum(pele_vals)/len(pele_vals) if pele_vals else None})

comp_strength = pd.DataFrame(rows).sort_values("mean_pele", ascending=False)
print(f"\nUnmatched squads (need adding to FBREF_TO_ELO): {sorted(unmatched)}")
print(f"\nComp strength (raw mean PELE):")
print(comp_strength.to_string(index=False))

# Normalize: divide by max so the strongest comp = 1.0
max_pele = comp_strength["mean_pele"].max()
comp_strength["strength_mult"] = comp_strength["mean_pele"] / max_pele
print(f"\nComp strength (normalized):")
print(comp_strength[["competition", "mean_pele", "strength_mult"]].to_string(index=False))


Sample ELO country names: ['Spain', 'Argentina', 'England', 'France', 'Brazil', 'Portugal', 'Germany', 'Netherlands', 'Colombia', 'Norway', 'Uruguay', 'Ecuador', 'Senegal', 'Italy', 'Turkey', 'Belgium', 'Switzerland', 'Croatia', 'Japan', 'Denmark']

Unmatched squads (need adding to FBREF_TO_ELO): ['Antigua', 'British Virgin Islands', 'Chinese Taipei', 'Equ. Guinea', 'Guadeloupe', 'Korea DPR', 'Kyrgyz Republic', 'Papua NG', 'St. Lucia', 'St. Vincent', 'São Tomé', 'Trin & Tobago', 'Turks & Caicos', 'Türkiye', 'US Virgin Islands']

Comp strength (raw mean PELE):
        competition  n_squads  n_matched   mean_pele
       CONMEBOL WCQ        10         10 1864.430000
          Euro 2024        24         23 1848.347826
  Copa America 2024        16         16 1820.731250
UEFA Nations League        54         53 1682.900000
           UEFA WCQ        54         53 1682.900000
      Gold Cup 2025        16         14 1655.064286
         AFCON 2025        24         23 1652.873913
          

In [23]:
# Save comp_strength so notebook 03 can pick it up
out = "../data/processed/comp_strength.csv"
comp_strength[["competition", "mean_pele", "strength_mult"]].to_csv(out, index=False)
print(f"Wrote {len(comp_strength)} comps → {out}")

Wrote 11 comps → ../data/processed/comp_strength.csv


## xMins Projections using International Data

In [169]:
# === Default xMins v4: hybrid normalization + mp_share ranking + WC-roster filter ===
import numpy as np
import unicodedata
import sys, os
sys.path.insert(0, os.path.abspath(".."))
from data.manual_overrides import MANUAL_OVERRIDES
from data.manual_starting_xi import MANUAL_STARTING_XI

def to_ascii(name):
    s = str(name)
    # Turkish dotless ı/İ → regular i/I (must happen before NFKD)
    s = s.replace("ı", "i").replace("İ", "I")
    for ch in ("'", "'", "`", "ʼ"):
        s = s.replace(ch, "")
    return unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode().lower().strip()


# ─── Load + per-player aggregates ───────────────────────────────────────────
intl = pd.read_csv("../data/processed/international_stats_2026.csv")
intl["min_num"] = pd.to_numeric(intl["Playing Time_Min"].astype(str).str.replace(",", ""), errors="coerce").fillna(0)
intl["mp_num"]  = pd.to_numeric(intl["Playing Time_MP"], errors="coerce").fillna(0)
intl["comp_w"]  = intl["competition"].map(COMP_WEIGHTS).fillna(0.5)

team_matches = (intl.groupby(["Squad", "competition"])["mp_num"].max().reset_index()
                .rename(columns={"mp_num": "team_matches"}))
team_matches["comp_w"] = team_matches["competition"].map(COMP_WEIGHTS).fillna(0.5)
team_matches["weighted_team_mp"] = team_matches["team_matches"] * team_matches["comp_w"]
team_avail = (team_matches.groupby("Squad")["weighted_team_mp"].sum()
              .reset_index().rename(columns={"weighted_team_mp": "team_mp_w"}))

intl["weighted_min"] = intl["min_num"] * intl["comp_w"]
intl["weighted_mp"]  = intl["mp_num"]  * intl["comp_w"]

player_agg = (intl.groupby(["Player", "Squad"], as_index=False)
              .agg(weighted_min=("weighted_min", "sum"),
                   weighted_mp=("weighted_mp", "sum")))

player_pos = (intl.groupby(["Player", "Squad"])["Pos"]
              .agg(lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else x.iloc[0])
              .reset_index())
player_agg = player_agg.merge(player_pos, on=["Player", "Squad"]).merge(team_avail, on="Squad")
player_agg["is_gk"]            = player_agg["Pos"].str.startswith("GK")
player_agg["mp_share"]         = player_agg["weighted_mp"] / player_agg["team_mp_w"]

K_COND_SHRINK = 5      # phantom appearances toward prior
PRIOR_COND    = 75     # typical starter minutes-per-appearance

player_agg["conditional_min_raw"] = np.where(
    player_agg["weighted_mp"] > 0,
    player_agg["weighted_min"] / player_agg["weighted_mp"], 0
)
player_agg["conditional_min"] = np.where(
    player_agg["weighted_mp"] > 0,
    (player_agg["weighted_min"] + K_COND_SHRINK * PRIOR_COND) /
    (player_agg["weighted_mp"] + K_COND_SHRINK),
    0
)
player_agg["per_team_match"]   = player_agg["weighted_min"] / player_agg["team_mp_w"]
player_agg["name_ascii"]       = player_agg["Player"].apply(to_ascii)

# ─── WC roster filter ───────────────────────────────────────────────────────
WC_TO_FBREF_SQUAD = {
    "Bosnia and Herzegovina": "Bosnia-Herzegovina",
    "Cabo Verde": "Cape Verde",
    "USA": "United States",
}
wc_roster = pd.read_csv("../data/processed/player_fixtures.csv")[["player", "team", "position"]].drop_duplicates()
wc_roster["name_ascii"] = wc_roster["player"].apply(
    lambda p: to_ascii(MANUAL_OVERRIDES.get(p, p))
)
wc_roster["fbref_squad"] = wc_roster["team"].map(WC_TO_FBREF_SQUAD).fillna(wc_roster["team"])

roster_keys = set(zip(wc_roster["name_ascii"], wc_roster["fbref_squad"]))
player_agg["roster_key"] = list(zip(player_agg["name_ascii"], player_agg["Squad"]))
player_agg["in_roster"]  = player_agg["roster_key"].isin(roster_keys)

print(f"Intl player rows: {len(player_agg)}")
print(f"  - in WC roster: {player_agg['in_roster'].sum()}")
print(f"  - NOT in roster (excluded): {(~player_agg['in_roster']).sum()}")

# ─── Pass 2 (enhanced): multi-scorer fuzzy match within-team ────────────────
from rapidfuzz import process, fuzz

matched_keys_so_far = set(zip(player_agg[player_agg["in_roster"]]["name_ascii"],
                              player_agg[player_agg["in_roster"]]["Squad"]))
roster_unmatched = wc_roster[~wc_roster.apply(
    lambda r: (r["name_ascii"], r["fbref_squad"]) in matched_keys_so_far, axis=1
)].copy()
unclaimed_intl = player_agg[~player_agg["in_roster"]][["Squad", "name_ascii"]].copy()

# Multiple scorers, each with its own threshold. Player matches if ANY passes.
# Within-team matching keeps false-positive risk low.
SCORERS = [
    ("token_sort_ratio", 80),  # was 80; relaxed for word-order/spelling variants
    ("WRatio",            85), # rapidfuzz's weighted heuristic — handles abbreviation
    ("partial_ratio",     90), # high threshold to require strong substring overlap
]

fuzzy_hits = []
match_log = []  # for inspection: (roster_name, matched_intl_name, scorer, score)

for _, r in roster_unmatched.iterrows():
    pool = unclaimed_intl[unclaimed_intl["Squad"] == r["fbref_squad"]]["name_ascii"].tolist()
    if not pool:
        continue

    best_match = None
    best_score = -1
    best_scorer = None
    for scorer_name, threshold in SCORERS:
        scorer = getattr(fuzz, scorer_name)
        result = process.extractOne(r["name_ascii"], pool,
                                     scorer=scorer, score_cutoff=threshold)
        if result:
            intl_ascii, score, _ = result
            if score > best_score:
                best_score = score
                best_match = intl_ascii
                best_scorer = scorer_name

    if best_match:
        fuzzy_hits.append((best_match, r["fbref_squad"]))
        match_log.append((r["player"], best_match, best_scorer, best_score))
        unclaimed_intl = unclaimed_intl[
            ~((unclaimed_intl["Squad"] == r["fbref_squad"]) &
              (unclaimed_intl["name_ascii"] == best_match))
        ]

print(f"Multi-scorer fuzzy matches: {len(fuzzy_hits)} (was 78 with single scorer)")
roster_keys.update(fuzzy_hits)
player_agg["in_roster"] = player_agg["roster_key"].isin(roster_keys)
print(f"Total matched after enhanced fuzzy: {player_agg['in_roster'].sum()}")

# Quick audit: top fuzzy hits by scorer (so you can spot suspicious matches)
import pandas as pd
log_df = pd.DataFrame(match_log, columns=["roster_name", "matched_intl", "scorer", "score"])
print(f"\nMatches by scorer:\n{log_df['scorer'].value_counts()}")
print(f"\nLowest-confidence matches (potential false positives, score < 80):")
print(log_df[log_df['score'] < 80].sort_values('score').head(15).to_string(index=False))


squad_agg = player_agg[player_agg["in_roster"]].copy()

# ─── Starter selection: min mp_share eligibility + per_team_match ranking ──
MIN_STARTER_MP_SHARE = 0.4

squad_agg["starter_eligible"] = squad_agg["mp_share"] >= MIN_STARTER_MP_SHARE
squad_agg["rank_in_pool"] = (
    squad_agg[squad_agg["starter_eligible"]]
    .groupby(["Squad", "is_gk"])["per_team_match"]
    .rank(method="first", ascending=False)
)
squad_agg["rank_in_pool"] = squad_agg["rank_in_pool"].fillna(999)
squad_agg["is_starter"] = (
    squad_agg["starter_eligible"] & (
        ((squad_agg["is_gk"]) & (squad_agg["rank_in_pool"] == 1)) |
        ((~squad_agg["is_gk"]) & (squad_agg["rank_in_pool"] <= 10))
    )
)


# ─── Manual XI overrides where defined (else keep algorithmic is_starter) ──
MANUAL_XI_ASCII = {
    team: {to_ascii(MANUAL_OVERRIDES.get(p, p)) for p in players}
    for team, players in MANUAL_STARTING_XI.items()
}
FBREF_TO_WC = {v: k for k, v in WC_TO_FBREF_SQUAD.items()}
squad_agg["wc_team"] = squad_agg["Squad"].map(FBREF_TO_WC).fillna(squad_agg["Squad"])

def _apply_manual_xi(row):
    team = row["wc_team"]
    if team in MANUAL_XI_ASCII:
        return row["name_ascii"] in MANUAL_XI_ASCII[team]
    return row["is_starter"]   # algorithmic fallback for teams without manual XI

squad_agg["is_starter"] = squad_agg.apply(_apply_manual_xi, axis=1)

# Audit: count how many manual-XI players matched per team
print("Manual XI match counts (should be 11 each for the 24 teams):")
for team, ascii_set in MANUAL_XI_ASCII.items():
    fbref_sq = WC_TO_FBREF_SQUAD.get(team, team)
    matched = squad_agg[(squad_agg["wc_team"] == team) & squad_agg["is_starter"]].shape[0]
    print(f"  {team:25s} {matched}/11")



# ─── Hybrid normalization (starters → conditional; non-starters → leftover) ─
starter_sum_gk = (squad_agg[squad_agg["is_gk"] & squad_agg["is_starter"]]
                  .groupby("Squad")["conditional_min"].sum().rename("starter_sum_gk").reset_index())
starter_sum_of = (squad_agg[~squad_agg["is_gk"] & squad_agg["is_starter"]]
                  .groupby("Squad")["conditional_min"].sum().rename("starter_sum_of").reset_index())
squad_agg = (squad_agg.merge(starter_sum_gk, on="Squad", how="left")
                       .merge(starter_sum_of, on="Squad", how="left"))
squad_agg["starter_sum"] = np.where(squad_agg["is_gk"], squad_agg["starter_sum_gk"], squad_agg["starter_sum_of"])
squad_agg["pool_budget"] = np.where(squad_agg["is_gk"], 90, 900)
squad_agg["starter_factor"] = np.minimum(1.0, squad_agg["pool_budget"] / squad_agg["starter_sum"])

ns_sum_gk = (squad_agg[squad_agg["is_gk"] & ~squad_agg["is_starter"]]
             .groupby("Squad")["per_team_match"].sum().rename("ns_sum_gk").reset_index())
ns_sum_of = (squad_agg[~squad_agg["is_gk"] & ~squad_agg["is_starter"]]
             .groupby("Squad")["per_team_match"].sum().rename("ns_sum_of").reset_index())
squad_agg = (squad_agg.merge(ns_sum_gk, on="Squad", how="left")
                       .merge(ns_sum_of, on="Squad", how="left"))
squad_agg["ns_sum"] = np.where(squad_agg["is_gk"], squad_agg["ns_sum_gk"], squad_agg["ns_sum_of"])
squad_agg["ns_sum"] = squad_agg["ns_sum"].fillna(0)
squad_agg["ns_budget"] = squad_agg["pool_budget"] - squad_agg["starter_sum"] * squad_agg["starter_factor"]
squad_agg["ns_factor"] = np.where(
    squad_agg["ns_sum"] > 0,
    np.minimum(1.0, squad_agg["ns_budget"] / squad_agg["ns_sum"]),
    0
)

squad_agg["xmins"] = np.where(
    squad_agg["is_starter"],
    squad_agg["conditional_min"] * squad_agg["starter_factor"],
    squad_agg["per_team_match"] * squad_agg["ns_factor"]
)

# ─── Build full output (incl unmatched roster) + even leftover distribution ─
# Map FBref Pos ("DF", "MF", "FW", "GK", "DF,MF" etc.) to our roster's position ("DEF", "MID", "FWD", "GK")
POS_MAP = {"GK": "GK", "DF": "DEF", "MF": "MID", "FW": "FWD"}

# Explode FBref's composite positions ("FW,MF" → two rows: FW + MF), map to roster format
squad_agg_exp = squad_agg.assign(
    pos_tokens=squad_agg["Pos"].str.split(",")
).explode("pos_tokens")
squad_agg_exp["pos_primary"] = squad_agg_exp["pos_tokens"].map(POS_MAP)
squad_agg_exp = squad_agg_exp.drop_duplicates(subset=["Player", "Squad", "pos_primary"])

# Merge on (name, squad, position) — composite-position players match against EITHER token
output = wc_roster[["player", "team", "position", "name_ascii", "fbref_squad"]].merge(
    squad_agg_exp[["name_ascii", "Squad", "pos_primary", "Pos", "is_starter", "xmins"]],
    left_on=["name_ascii", "fbref_squad", "position"],
    right_on=["name_ascii", "Squad", "pos_primary"],
    how="left"
).drop_duplicates(subset=["player", "team", "position"], keep="first")

output["xmins"]      = output["xmins"].fillna(0)
output["is_starter"] = output["is_starter"].fillna(False)

matched = (output["xmins"] > 0).sum()
print(f"Position-aware match (composite-expanded): {matched} / {len(output)} matched")



# Distribute leftover budget evenly across squad (capped at 90 per player)
team_total = output.groupby("team")["xmins"].sum().rename("team_total").reset_index()
team_size  = output.groupby("team").size().rename("team_size").reset_index()
output = output.merge(team_total, on="team").merge(team_size, on="team")
output["leftover"]    = (990 - output["team_total"]).clip(lower=0)
output["even_boost"]  = output["leftover"] / output["team_size"]
output["xmins"]       = (output["xmins"] + output["even_boost"]).clip(upper=90)

# Sanity check: team totals after redistribution
print("Per-team total xMins after even-distribution fallback:")
totals_after = output.groupby("team")["xmins"].sum().sort_values()
print(totals_after.head()); print("..."); print(totals_after.tail())

# Spot-check the affected teams
for sq in ["Brazil", "Egypt", "Jordan", "England", "Spain"]:
    print(f"\n{sq} top 15 by xMins after redistribution:")
    df = output[output["team"] == sq].nlargest(15, "xmins")
    cols = ["player", "Pos", "is_starter", "xmins"]
    print(df[cols].round(2).to_string(index=False))


Intl player rows: 1944
  - in WC roster: 1025
  - NOT in roster (excluded): 919
Multi-scorer fuzzy matches: 97 (was 78 with single scorer)
Total matched after enhanced fuzzy: 1122

Matches by scorer:
scorer
token_sort_ratio    52
WRatio              35
partial_ratio       10
Name: count, dtype: int64

Lowest-confidence matches (potential false positives, score < 80):
Empty DataFrame
Columns: [roster_name, matched_intl, scorer, score]
Index: []
Manual XI match counts (should be 11 each for the 24 teams):
  Mexico                    9/11
  Czechia                   11/11
  Canada                    11/11
  Switzerland               11/11
  Brazil                    11/11
  Morocco                   11/11
  Scotland                  11/11
  USA                       11/11
  Türkiye                   11/11
  Germany                   11/11
  Côte d'Ivoire             11/11
  Ecuador                   9/11
  Netherlands               11/11
  Japan                     11/11
  Sweden         

Export xMins

In [170]:
# === Export default_xmins.csv for build_projections.py ===
default_xmins = output[["player", "team", "position", "xmins"]].rename(
    columns={"xmins": "default_xmins"}
)
default_xmins["default_xmins"] = default_xmins["default_xmins"].round(2)

out_path = "../data/default_xmins.csv"
default_xmins.to_csv(out_path, index=False)

print(f"Wrote {len(default_xmins)} rows → {out_path}")
print(f"\nSample:")
print(default_xmins.head(10).to_string(index=False))

# Per-team summary (sanity check)
print(f"\nTeam total xMins summary (5 lowest, 5 highest):")
team_summary = (default_xmins.groupby("team")["default_xmins"]
                .agg(team_total="sum", n_players="count")
                .sort_values("team_total"))
print(team_summary.head())
print("...")
print(team_summary.tail())

# Distribution check
print(f"\nxMins distribution:")
print(default_xmins["default_xmins"].describe().round(2))

Wrote 1454 rows → ../data/default_xmins.csv



Sample:
          player    team position  default_xmins
 Rayan Aït-Nouri Algeria      DEF          89.87
 Ramy Bensebaini Algeria      DEF          90.00
     Aïssa Mandi Algeria      DEF          90.00
    Mehdi Dorval Algeria      DEF          14.76
Zinéddine Belaïd Algeria      DEF          20.32
     Sohaib Nair Algeria      DEF          10.56
  Rafik Belghali Algeria      DEF          86.11
    Achref Abada Algeria      DEF          10.56
 Mohammed Amoura Algeria      FWD          10.56
    Amine Gouiri Algeria      FWD          74.84

Team total xMins summary (5 lowest, 5 highest):
            team_total  n_players
team                             
IR Iran         959.48         30
Jordan          964.02         30
Egypt           965.08         27
Tunisia         966.92         26
Cabo Verde      967.70         26
...
             team_total  n_players
team                              
Ghana            990.04         28
New Zealand      990.05         26
Czechia          990

## Manual Starters Validation

In [164]:
# === Preview function: validate XI + show full team xMins given those starters ===
from rapidfuzz import process, fuzz

def preview_manual_xi(team, proposed_players):
    """Validate proposed XI and preview the resulting xMins for all squad members."""
    fbref_squad = WC_TO_FBREF_SQUAD.get(team, team)
    team_data = squad_agg[squad_agg["Squad"] == fbref_squad].copy()
    team_roster = wc_roster[wc_roster["team"] == team].copy()

    if team_data.empty:
        print(f"⚠ No intl data for team '{team}'")
        return

    # Validation
    ascii_to_fbref = dict(zip(team_data["name_ascii"], team_data["Player"]))
    proposed_ascii, unmatched = [], []
    for p in proposed_players:
        a = to_ascii(MANUAL_OVERRIDES.get(p, p))
        if a in ascii_to_fbref:
            proposed_ascii.append(a)
        else:
            r = process.extractOne(a, list(ascii_to_fbref.keys()),
                                    scorer=fuzz.WRatio, score_cutoff=55)
            unmatched.append((p, ascii_to_fbref[r[0]] if r else None, r[1] if r else None))

    if unmatched:
        print(f"⚠ {len(unmatched)}/{len(proposed_players)} proposed players unmatched:")
        for name, sug, score in unmatched:
            sug_str = f"did you mean '{sug}'? (score {score:.0f})" if sug else "no close match"
            print(f"  ✗ {name:30s} → {sug_str}")
        print("(preview below assumes the matched ones as starters)\n")

    # Override is_starter for this team
    team_data["is_starter"] = team_data["name_ascii"].isin(proposed_ascii)

    # Hybrid normalization, scoped to this team
    for is_gk_val in [True, False]:
        mask = team_data["is_gk"] == is_gk_val
        starters     = team_data[mask & team_data["is_starter"]]
        non_starters = team_data[mask & ~team_data["is_starter"]]
        budget = 90 if is_gk_val else 900
        sst = starters["conditional_min"].sum()
        starter_factor = min(1.0, budget / sst) if sst > 0 else 0
        ns_budget = budget - sst * starter_factor
        ns_sum = non_starters["per_team_match"].sum()
        ns_factor = min(1.0, ns_budget / ns_sum) if ns_sum > 0 else 0
        team_data.loc[mask & team_data["is_starter"], "xmins"] = (
            team_data.loc[mask & team_data["is_starter"], "conditional_min"] * starter_factor)
        team_data.loc[mask & ~team_data["is_starter"], "xmins"] = (
            team_data.loc[mask & ~team_data["is_starter"], "per_team_match"] * ns_factor)


    # Hybrid merge: name-only for unique names, position-aware for collisions (e.g., Danilos)
    POS_MAP = {"GK": "GK", "DF": "DEF", "MF": "MID", "FW": "FWD"}
    te = team_data.assign(pos_tokens=team_data["Pos"].str.split(",")).explode("pos_tokens")
    te["pos_primary"] = te["pos_tokens"].map(POS_MAP)
    te = te.drop_duplicates(subset=["Player", "pos_primary"])

    team_roster["dup_count"] = team_roster.groupby("name_ascii")["name_ascii"].transform("count")

    # Unique names → name-only merge (handles position discrepancy)
    te_namekey = te.drop_duplicates(subset="name_ascii")
    out_uniq = team_roster[team_roster["dup_count"] == 1].merge(
        te_namekey[["name_ascii", "is_starter", "xmins", "mp_share", "conditional_min"]],
        on="name_ascii", how="left"
    )

    # Duplicate names (Danilos) → position-aware merge to disambiguate
    out_dup = team_roster[team_roster["dup_count"] > 1].merge(
        te[["name_ascii", "pos_primary", "is_starter", "xmins", "mp_share", "conditional_min"]],
        left_on=["name_ascii", "position"], right_on=["name_ascii", "pos_primary"], how="left"
    ).drop_duplicates(subset=["player", "position"], keep="first")

    out = pd.concat([out_uniq, out_dup], ignore_index=True)
    out["xmins"] = out["xmins"].fillna(0)
    out["is_starter"] = out["is_starter"].fillna(False)
    out["mp_share"] = out["mp_share"].fillna(0)
    out["conditional_min"] = out["conditional_min"].fillna(0)

    leftover = max(0, 990 - out["xmins"].sum())
    boost = leftover / len(out) if len(out) > 0 else 0
    out["xmins"] = (out["xmins"] + boost).clip(upper=90)
    
    
    print(f"=== {team} preview ({len(out)} squad players, total xMins {out['xmins'].sum():.1f}) ===\n")
    sorted_out = out.sort_values(["is_starter", "xmins"], ascending=[False, False])
    cols = ["player", "position", "is_starter", "mp_share", "conditional_min", "xmins"]
    print(sorted_out[cols].round(2).to_string(index=False))

### Group A

Mexico

In [100]:
preview_manual_xi("Mexico", [
    "Raúl Rangel",   # GK
    "Israel Reyes",       # RB
    "Cesar Montes",       # CB
    "Johan Vasquez",        # CB
    "Jesus Gallardo",        # LB
    "Edson Alvarez",       # CDM
    "Erik Lira",   # CM
    "Álvaro Fidalgo",  # CM
    "Roberto Alvarado",       # RW
    "Raul Jimenez",        # ST
    "Julian Quinones",    # LW
])

⚠ 2/11 proposed players unmatched:
  ✗ Raúl Rangel                    → did you mean 'Raúl Jiménez'? (score 61)
  ✗ Álvaro Fidalgo                 → did you mean 'Edson Álvarez'? (score 56)
(preview below assumes the matched ones as starters)

=== Mexico preview (51 squad players, total xMins 990.0) ===

            player position is_starter  mp_share  conditional_min  xmins
     Johan Vásquez      DEF       True      0.88            81.59  83.63
      César Montes      DEF       True      0.88            79.90  81.94
     Edson Álvarez      MID       True      0.83            79.45  81.49
  Roberto Alvarado      FWD       True      0.83            75.20  77.24
      Raúl Jiménez      FWD       True      0.75            74.68  76.73
    Jesús Gallardo      DEF       True      0.62            72.86  74.90
         Érik Lira      MID       True      0.38            70.45  72.49
      Israel Reyes      DEF       True      0.46            64.84  66.88
   Julián Quiñones      FWD       Tru

Czech Republic

In [167]:
preview_manual_xi("Czechia", [
    "Matej Kovar",   # GK
    "Štěpán Chaloupek",       # RB
    "Robin Hranáč",       # CB
    "Ladislav Krejci",        # CB
    "Vladimir Coufal",        # LB
    "Tomas Soucek",       # CDM
    "Vladimír Darida",   # CM
    "Jaroslav Zelený",  # CM
    "Lukáš Provod",       # RW
    "Pavel Šulc",        # ST
    "Patrik Schick",    # LW
])

=== Czechia preview (29 squad players, total xMins 990.0) ===

            player position is_starter  mp_share  conditional_min  xmins
   Ladislav Krejcí      DEF       True      0.80            89.15  89.15
       Matej Kovár       GK       True      0.88            87.79  87.79
   Vladimír Coufal      DEF       True      0.94            86.75  86.75
      Tomás Soucek      MID       True      0.95            85.90  85.90
      Robin Hranác      DEF       True      0.39            84.40  84.40
   Jaroslav Zeleny      DEF       True      0.59            80.91  80.91
        Pavel Sulc      MID       True      0.88            79.45  79.45
     Patrik Schick      FWD       True      0.66            75.24  75.24
  Stepán Chaloupek      DEF       True      0.19            74.12  74.12
      Lukás Provod      MID       True      0.94            71.71  71.71
   Vladimír Darida      MID       True      0.12            68.43  68.43
        Lukás Cerv      MID      False      0.73            7

### Group B

Canada

In [101]:
preview_manual_xi("Canada", [
    "Dayne St. Clair",   # GK
    "Alistair Johnston",       # RB
    "Moïse Bombito",       # CB
    "Derek Cornelius",        # CB
    "Richie Laryea",        # LB
    "Tajon Buchanan",       # CDM
    "Ismael Kone",   # CM
    "Stephen Eustaquio",  # CM
    "Ali Ahmed",       # RW
    "Jonathan David",        # ST
    "Cyle Larin",    # LW
])

=== Canada preview (32 squad players, total xMins 990.0) ===

            player position is_starter  mp_share  conditional_min  xmins
     Moïse Bombito      DEF       True      0.50            80.62  80.62
   Dayne St. Clair       GK       True      0.46            80.32  80.32
 Stephen Eustaquio      MID       True      0.42            78.73  78.73
    Jonathan David      FWD       True      1.00            74.61  74.61
 Alistair Johnston      DEF       True      0.88            73.73  73.73
   Derek Cornelius      DEF       True      0.75            73.58  73.58
     Richie Laryea      DEF       True      0.88            72.15  72.15
       Ismaël Koné      MID       True      0.67            67.97  67.97
         Ali Ahmed      MID       True      0.50            64.81  64.81
    Tajon Buchanan      FWD       True      0.75            64.13  64.13
        Cyle Larin      FWD       True      0.92            60.21  60.21
 Jacob Shaffelburg      MID      False      1.00            65

Switzerland

In [102]:
preview_manual_xi("Switzerland", [
    "Gregor Kobel",   # GK
    "Silvan Widmer",       # RB
    "Manuel Akanji",       # CB
    "Nico Elvedi",        # CB
    "Ricardo Rodriguez",        # LB
    "Granit Xhaka",       # CDM
    "Remo Freuler",   # CM
    "Denis Zakaria",        # CM
    "Dan Ndoye",       # RW
    "Breel Embolo",        # ST
    "Ruben Vargas",    # LW
])

=== Switzerland preview (26 squad players, total xMins 990.0) ===

             player position is_starter  mp_share  conditional_min  xmins
      Manuel Akanji      DEF       True      0.88            86.36  86.36
       Remo Freuler      MID       True      0.85            85.03  85.03
       Gregor Kobel       GK       True      0.75            84.92  84.92
       Granit Xhaka      MID       True      0.94            84.13  84.13
          Dan Ndoye      FWD       True      0.69            82.43  82.43
        Nico Elvedi      DEF       True      0.63            82.13  82.13
  Ricardo Rodríguez      DEF       True      0.94            77.94  77.94
       Breel Embolo      FWD       True      0.94            75.45  75.45
      Silvan Widmer      DEF       True      0.73            72.84  72.84
       Rubén Vargas      MID       True      0.77            70.73  70.73
      Denis Zakaria      MID       True      0.31            53.42  53.42
   Michel Aebischer      MID      False      

### Group C

Brazil

In [103]:
preview_manual_xi("Brazil", [
    "Alisson",   # GK
    "Wesley",       # RB
    "Marquinhos",       # CB
    "Gabriel Magalhães",        # CB
    "Alex Sandro",        # LB
    "Bruno Guimaraes",       # CDM
    "Casemiro",   # CM
    "Matheus Cunha",        # CM
    "Raphinha",       # RW
    "Vinicius Júnior",        # ST
    "Endrick",    # LW
])

=== Brazil preview (26 squad players, total xMins 990.0) ===

            player position is_starter  mp_share  conditional_min  xmins
    Alisson Becker       GK       True      0.55            84.50  86.73
        Marquinhos      DEF       True      1.00            82.80  85.03
 Gabriel Magalhães      DEF       True      0.72            82.15  84.38
          Casemiro      MID       True      0.35            80.67  82.89
   Vinícius Júnior      MID       True      0.62            79.60  81.83
          Raphinha      MID       True      0.75            77.78  80.00
   Bruno Guimarães      MID       True      0.95            77.25  79.48
       Alex Sandro      DEF       True      0.10            76.86  79.08
            Wesley      DEF       True      0.15            70.88  73.10
     Matheus Cunha      FWD       True      0.35            51.50  53.73
           Endrick      FWD       True      0.40            44.96  47.19
            Danilo      DEF      False      0.50            75

Morocco

In [143]:
preview_manual_xi("Morocco", [
    "Yassine Bounou",   # GK
    "Achraf Hakimi",       # RB
    "Nayef Aguerd",       # CB
    "Chadi Riad",        # CB
    "Noussair Mazraoui",        # LB
    "Sofyan Amrabat",       # CDM
    "Neil El Aynaoui",   # CM
    "Azzedine Ounahi",        # CM
    "Brahim Diaz",       # RW
    "Ayoub El Kaabi",        # ST
    "Abde Ezzalzouli",    # LW
])

=== Morocco preview (46 squad players, total xMins 990.0) ===

                     player position is_starter  mp_share  conditional_min  xmins
             Yassine Bounou       GK       True      1.00            88.36  88.39
               Nayef Aguerd      DEF       True      0.92            88.26  88.30
            Neil El Aynaoui      MID       True      0.62            87.74  87.77
              Achraf Hakimi      DEF       True      0.74            84.76  84.80
          Noussair Mazraoui      DEF       True      0.62            84.06  84.09
                Brahim Díaz      MID       True      0.85            80.74  80.77
             Sofyan Amrabat      MID       True      0.62            79.06  79.09
                 Chadi Riad      DEF       True      0.08            77.50  77.54
            Abde Ezzalzouli      FWD       True      0.70            66.72  66.76
            Azzedine Ounahi      MID       True      0.55            63.20  63.24
             Ayoub El Kaabi      FW

Scotland

In [165]:
preview_manual_xi("Scotland", [
    "Angus Gunn",   # GK
    "Grant Hanley",       # RB
    "Jack Hendry",       # CB
    "Kieran Tierney",        # CB
    "Aaron Hickey",        # LB
    "Billy Gilmour",       # CDM
    "Scott McTominay",   # CM
    "Andrew Robertson",        # CM
    "John McGinn",       # RW
    "Ryan Christie",        # ST
    "Che Adams",    # LW
])

=== Scotland preview (26 squad players, total xMins 990.0) ===

            player position is_starter  mp_share  conditional_min  xmins
    Andy Robertson      DEF       True      1.00            85.47  85.47
   Scott McTominay      MID       True      1.00            84.87  84.87
        Angus Gunn       GK       True      0.58            83.75  83.75
       John McGinn      MID       True      0.88            80.63  80.63
      Grant Hanley      DEF       True      0.83            80.18  80.18
       Jack Hendry      DEF       True      0.21            76.87  76.87
     Billy Gilmour      MID       True      0.75            75.95  75.95
         Ché Adams      FWD       True      0.75            70.89  70.89
      Aaron Hickey      DEF       True      0.42            66.40  66.40
     Ryan Christie      MID       True      0.92            65.78  65.78
    Kieran Tierney      DEF       True      0.33            54.33  54.33
      John Souttar      DEF      False      0.67            

### Group D

United States

In [ ]:
preview_manual_xi("USA", [
    "Matt Freese",   # GK
    "Alexander Freeman",       # RB
    "Chris Richards",       # CB
    "Tim Ream",        # CB
    "Antonee Robinson",        # LB
    "Tyler Adams",       # CDM
    "Weston McKennie",   # CM
    "Malik Tillman",        # CM
    "Timothy Weah",       # RW
    "Folarin Balogun",        # ST
    "Christian Pulisic",    # LW
])

=== USA preview (31 squad players, total xMins 990.0) ===

             player position is_starter  mp_share  conditional_min  xmins
     Chris Richards      DEF       True      1.00            82.91  82.91
           Tim Ream      DEF       True      1.00            82.86  82.86
        Matt Freese       GK       True      0.75            82.11  82.11
  Alexander Freeman      DEF       True      0.75            81.87  81.87
   Antonee Robinson      DEF       True      0.25            78.46  78.46
  Christian Pulisic      MID       True      0.25            78.46  78.46
      Malik Tillman      MID       True      0.83            78.10  78.10
    Weston McKennie      MID       True      0.25            77.46  77.46
        Tyler Adams      MID       True      0.88            71.93  71.93
    Folarin Balogun      FWD       True      0.25            71.15  71.15
       Timothy Weah      MID       True      0.17            71.00  71.00
   Patrick Agyemang      FWD      False      0.75    

Turkey

In [105]:
preview_manual_xi("Türkiye", [
    "Uğurcan Çakır",       # GK
    "Zeki Çelik",          # RB
    "Merih Demiral",       # CB
    "Abdülkerim Bardakcı", # CB
    "Ferdi Kadioglu",      # LB
    "Hakan Çalhanoglu",    # CDM
    "İsmail Yüksek",       # CM
    "Barış Alper Yılmaz",  # CM
    "Arda Güler",          # RW
    "Kenan Yıldız",        # ST
    "Kerem Aktürkoğlu",    # LW
])

=== Türkiye preview (35 squad players, total xMins 990.0) ===

             player position is_starter  mp_share  conditional_min  xmins
Abdülkerim Bardakci      DEF       True      0.85            85.77  86.05
      Ugurcan Çakir       GK       True      0.57            84.44  84.72
         Arda Güler      MID       True      0.93            79.26  79.53
      Merih Demiral      DEF       True      0.78            77.40  77.67
   Hakan Çalhanoglu      MID       True      0.85            75.00  75.27
       Kenan Yildiz      MID       True      0.88            73.86  74.13
      Ismail Yüksek      MID       True      0.70            71.82  72.09
   Kerem Aktürkoglu      FWD       True      0.93            69.58  69.85
     Ferdi Kadioglu      DEF       True      0.80            69.38  69.65
         Zeki Çelik      DEF       True      0.68            63.23  63.50
 Baris Alper Yilmaz      FWD       True      0.70            61.76  62.03
        Orkun Kökçü      MID      False      0.90

### Group E

Germany

In [66]:
preview_manual_xi("Germany", [
    "Oliver Baumann",   # GK
    "Joshua Kimmich",       # RB
    "Jonathan Tah",       # CB
    "Nico Schlotterbeck",        # CB
    "David Raum",        # LB
    "Aleksandar Pavlovic",       # CDM
    "Leon Goretzka",   # CM
    "Florian Wirtz",        # CM
    "Leroy Sane",       # RW
    "Jamal Musiala",        # ST
    "Kai Havertz",    # LW
])

=== Germany preview (26 squad players, total xMins 990.0) ===

             player position is_starter  mp_share  conditional_min  xmins
     Oliver Baumann       GK       True      0.56            84.64  84.64
     Joshua Kimmich      DEF       True      0.94            83.29  83.29
       Jonathan Tah      DEF       True      0.88            78.30  78.30
      Florian Wirtz      MID       True      0.91            75.69  75.69
      Jamal Musiala      MID       True      0.44            75.65  75.65
        Kai Havertz      FWD       True      0.34            75.60  75.60
      Leon Goretzka      MID       True      0.56            74.46  74.46
 Nico Schlotterbeck      DEF       True      0.53            73.22  73.22
         David Raum      DEF       True      0.66            72.60  72.60
Aleksandar Pavlovic      MID       True      0.44            69.98  69.98
         Leroy Sané      MID       True      0.52            66.66  66.66
    Antonio Rüdiger      DEF      False      0.52

Cote d'Ivoire

In [137]:
preview_manual_xi("Côte d'Ivoire", [
    "Yahia Fofana",   # GK
    "Wilfried Singo",       # RB
    "Ousmane Diomande",       # CB
    "Obite N'Dicka",        # CB
    "Ghislain Konan",        # LB
    "Franck Kessie",       # CDM
    "Ibrahim Sangare",   # CM
    "Seko Fofana",        # CM
    "Amad Diallo",       # RW
    "Yan Diomande",        # ST
    "Evann Guessand",    # LW
])

=== Côte d'Ivoire preview (26 squad players, total xMins 990.0) ===

           player position is_starter  mp_share  conditional_min  xmins
     Yahia Fofana       GK       True      0.87            85.59  86.09
      Evan Ndicka      DEF       True      0.73            85.00  85.51
   Ghislain Konan      DEF       True      0.58            83.85  84.35
    Franck Kessie      MID       True      1.00            77.89  78.40
      Amad Diallo      FWD       True      0.49            73.47  73.97
     Yan Diomande      FWD       True      0.42            72.30  72.81
  Ibrahim Sangaré      MID       True      0.73            71.57  72.07
 Ousmane Diomande      DEF       True      0.18            71.10  71.61
   Wilfried Singo      DEF       True      0.44            69.73  70.23
      Seko Fofana      MID       True      0.65            62.57  63.08
   Evann Guessand      FWD       True      0.65            59.70  60.20
 Odilon Kossounou      DEF      False      0.44            83.11  3

Ecuador

In [139]:
preview_manual_xi("Ecuador", [
    "Hernan Galindez",   # GK
    "Joel Ordóñez",       # RB
    "Willian Pacho",       # CB
    "Piero Hincapie",        # CB
    "Alan Franco",        # LB
    "Moises Caicedo",       # CDM
    "Pedro Vite",   # CM
    "Pervis Estupinan",        # CM
    "Gonzalo Plata",       # RW
    "John Yeboah",        # ST
    "Enner Valencia",    # LW
])

⚠ 2/11 proposed players unmatched:
  ✗ Willian Pacho                  → did you mean 'Alan Franco'? (score 58)
  ✗ Piero Hincapie                 → did you mean 'Pedro Vite'? (score 58)
(preview below assumes the matched ones as starters)

=== Ecuador preview (34 squad players, total xMins 990.0) ===

          player position is_starter  mp_share  conditional_min  xmins
 Hernán Galíndez       GK       True      0.60            85.59  89.13
  Moisés Caicedo      MID       True      0.90            84.96  88.50
Pervis Estupiñán      DEF       True      0.65            82.94  86.49
    Joel Ordóñez      DEF       True      0.40            82.00  85.54
     Alan Franco      MID       True      0.75            80.20  83.74
      Pedro Vite      MID       True      0.45            76.43  79.97
   Gonzalo Plata      FWD       True      0.45            72.00  75.54
  Enner Valencia      FWD       True      0.82            71.02  74.57
     John Yeboah      MID       True      0.52            

### Group F

Netherlands

In [ ]:
preview_manual_xi("Netherlands", [
    "Bart Verbruggen",   # GK
    "Denzel Dumfries",       # RB
    "Jan Paul Van Hecke",       # CB
    "Virgil Van Dijk",        # CB
    "Micky Van de Ven",        # LB
    "Ryan Gravenberch",       # CDM
    "Tijjani Reijnders",   # CM
    "Frenkie de Jong",        # CM
    "Donyell Malen",       # RW
    "Memphis",        # ST
    "Cody Gakpo",    # LW
])

=== Netherlands preview (34 squad players, total xMins 990.0) ===

               player position is_starter  mp_share  conditional_min  xmins
      Bart Verbruggen       GK       True      0.84            87.27  89.41
      Virgil van Dijk      DEF       True      0.91            87.00  89.14
      Denzel Dumfries      DEF       True      0.72            83.88  86.02
      Frenkie de Jong      MID       True      0.54            79.09  81.23
           Cody Gakpo      FWD       True      1.00            77.74  79.88
    Tijjani Reijnders      MID       True      0.90            75.47  77.61
     Ryan Gravenberch      MID       True      0.62            74.73  76.87
   Jan Paul van Hecke      DEF       True      0.41            67.56  69.70
     Micky van de Ven      DEF       True      0.62            66.19  68.33
        Donyell Malen      FWD       True      0.79            56.65  58.79
          Xavi Simons      MID      False      0.79            63.01  24.92
       Stefan de Vrij

Japan

In [109]:
preview_manual_xi("Japan", [
    "Zion Suzuki",   # GK
    "Takehiro Tomiyasu",       # RB
    "Hiroki Ito",       # CB
    "Ko Itakura",        # CB
    "Ritsu Doan",        # LB
    "Wataru Endo",       # CDM
    "Ao Tanaka",   # CM
    "Keito Nakamura",        # CM
    "Takefusa Kubo",       # RW
    "Daichi Kamada",        # ST
    "Ayase Ueda",    # LW
])

=== Japan preview (27 squad players, total xMins 990.0) ===

           player position is_starter  mp_share  conditional_min  xmins
      Zion Suzuki       GK       True      0.83            85.00  85.00
       Ko Itakura      DEF       True      0.92            79.12  79.12
       Hiroki Ito      DEF       True      0.50            79.09  79.09
Takehiro Tomiyasu      DEF       True      0.17            77.14  77.14
      Wataru Endo      MID       True      0.92            77.12  77.12
       Ayase Ueda      FWD       True      0.75            75.71  75.71
        Ao Tanaka      MID       True      0.75            69.29  69.29
       Ritsu Doan      DEF       True      1.00            64.47  64.47
    Takefusa Kubo      MID       True      0.92            64.25  64.25
    Daichi Kamada      MID       True      1.00            60.76  60.76
   Keito Nakamura      MID       True      0.83            52.80  52.80
  Shogo Taniguchi      DEF      False      0.67            75.15  43.34
   

Sweden

In [111]:
preview_manual_xi("Sweden", [
    "Kristoffer Nordfeldt",   # GK
    "Isak Hien",       # RB
    "Carl Starfelt",       # CB
    "Victor Lindelof",        # CB
    "Daniel Svensson",        # LB
    "Yasin Ayari",       # CDM
    "Jesper Karlstrom",   # CM
    "Gabriel Gudmundsson",        # CM
    "Anthony Elanga",       # RW
    "Viktor Gyokeres",        # ST
    "Alexander Isak",    # LW
])

=== Sweden preview (26 squad players, total xMins 990.0) ===

                  player position is_starter  mp_share  conditional_min  xmins
         Viktor Gyökeres      FWD       True      0.84            84.48  84.48
               Isak Hien      DEF       True      0.78            81.25  81.25
    Kristoffer Nordfeldt       GK       True      0.16            79.29  79.29
     Gabriel Gudmundsson      DEF       True      0.94            77.31  77.31
             Yasin Ayari      MID       True      0.92            74.03  74.03
         Victor Lindelöf      DEF       True      0.44            73.19  73.19
        Jesper Karlström      MID       True      0.50            72.47  72.47
           Carl Starfelt      DEF       True      0.42            72.15  72.15
          Alexander Isak      FWD       True      0.56            72.00  72.00
          Anthony Elanga      FWD       True      0.60            66.46  66.46
         Daniel Svensson      DEF       True      0.70            65.

### Group G

Belgium

In [113]:
preview_manual_xi("Belgium", [
    "Thibaut Courtois",   # GK
    "Timothy Castagne",       # RB
    "Zeno Debast",       # CB
    "Arthur Theate",        # CB
    "Maxim De Cuyper",        # LB
    "Amadou Onana",       # CDM
    "Youri Tielemans",   # CM
    "Kevin de Bruyne",        # CM
    "Charles de Ketelaere",       # RW
    "Romelu Lukaku",        # ST
    "Jeremy Doku",    # LW
])

=== Belgium preview (26 squad players, total xMins 990.0) ===

                player position is_starter  mp_share  conditional_min  xmins
      Thibaut Courtois       GK       True      0.28            81.67  81.75
         Romelu Lukaku      FWD       True      0.33            81.33  81.42
       Kevin De Bruyne      MID       True      0.66            79.52  79.60
           Jérémy Doku      MID       True      0.90            79.18  79.26
           Zeno Debast      DEF       True      0.79            78.24  78.32
         Arthur Theate      DEF       True      0.91            76.59  76.67
      Timothy Castagne      DEF       True      0.86            73.91  74.00
          Amadou Onana      MID       True      0.64            72.88  72.96
       Youri Tielemans      MID       True      0.72            68.35  68.44
       Maxim De Cuyper      DEF       True      0.69            65.35  65.43
  Charles De Ketelaere      MID       True      0.59            63.15  63.23
      Leandro

### Group H

Spain

In [115]:
preview_manual_xi("Spain", [
    "Unai Simon",   # GK
    "Marcos Llorente",       # RB
    "Pau Cubarsi",       # CB
    "Aymeric Laporte",        # CB
    "Marc Cucurella",        # LB
    "Rodri",       # CDM
    "Pedri",   # CM
    "Fabián Ruiz",        # CM
    "Ferran Torres",       # RW
    "Mikel Oyarzabal",        # ST
    "Nico Williams",    # LW
])

=== Spain preview (26 squad players, total xMins 990.0) ===

            player position is_starter  mp_share  conditional_min  xmins
        Unai Simón       GK       True      0.71            89.12  89.49
    Marc Cucurella      DEF       True      0.78            87.05  87.42
   Aymeric Laporte      DEF       True      0.57            84.41  84.78
       Fabián Ruiz      MID       True      0.74            75.54  75.91
     Nico Williams      MID       True      0.65            73.02  73.38
       Pau Cubarsí      DEF       True      0.37            70.60  70.97
             Rodri      MID       True      0.34            67.49  67.86
   Marcos Llorente      DEF       True      0.18            65.75  66.12
   Mikel Oyarzabal      FWD       True      0.91            65.02  65.39
             Pedri      MID       True      0.79            59.88  60.25
     Ferran Torres      FWD       True      0.51            57.67  58.04
  Martín Zubimendi      MID      False      0.81            74.

Uruguay

In [120]:
preview_manual_xi("Uruguay", [
    "Sergio Rochet",   # GK
    "Guillermo Varela",       # RB
    "Ronald Araujo",       # CB
    "Jose Maria Gimenez",        # CB
    "Mathias Olivera",        # LB
    "Manuel Ugarte",       # CDM
    "Federico Valverde",   # CM
    "Nicolás de la Cruz",        # CM
    "Agustin Canobbio",       # RW
    "Maximiliano Araújo",        # ST
    "Darwin Nunez",    # LW
])

=== Uruguay preview (32 squad players, total xMins 990.0) ===

                player position is_starter  mp_share  conditional_min  xmins
         Sergio Rochet       GK       True      0.85            86.59  86.62
     Federico Valverde      MID       True      0.90            83.72  83.74
         Ronald Araujo      DEF       True      0.50            81.57  81.59
         Manuel Ugarte      MID       True      0.95            78.60  78.63
    José María Giménez      DEF       True      0.57            78.24  78.27
           Maxi Araújo      MID       True      0.90            76.87  76.89
          Darwin Núñez      FWD       True      0.80            75.24  75.26
       Mathías Olivera      DEF       True      0.72            74.21  74.23
    Nicolás de la Cruz      MID       True      0.52            72.13  72.15
      Guillermo Varela      DEF       True      0.45            64.00  64.02
      Agustín Canobbio      FWD       True      0.18            53.12  53.14
     Facundo 

### Group J

Austria

In [122]:
preview_manual_xi("Austria", [
    "Alexander Schlager",   # GK
    "Konrad Laimer",       # RB
    "Kevin Danso",       # CB
    "David Alaba",        # CB
    "Phillipp Mwene",        # LB
    "Nicolas Seiwald",       # CDM
    "Xaver Schlager",   # CM
    "Patrick Wimmer",        # CM
    "Christoph Baumgartner",       # RW
    "Marcel Sabitzer",        # ST
    "Marko Arnautovic",    # LW
])

=== Austria preview (26 squad players, total xMins 990.0) ===

               player position is_starter  mp_share  conditional_min  xmins
      Nicolas Seiwald      MID       True      1.00            85.44  85.44
      Marcel Sabitzer      MID       True      0.95            84.93  84.93
   Alexander Schlager       GK       True      0.45            83.48  83.48
        Konrad Laimer      MID       True      1.00            79.62  79.62
Christoph Baumgartner      MID       True      0.93            75.55  75.55
       Phillipp Mwene      DEF       True      0.71            74.67  74.67
          David Alaba      DEF       True      0.28            74.67  74.67
          Kevin Danso      DEF       True      0.57            66.08  66.08
     Marko Arnautovic      FWD       True      0.88            64.82  64.82
       Xaver Schlager      MID       True      0.34            64.10  64.10
       Patrick Wimmer      MID       True      0.76            52.52  52.52
     Philipp Lienhart    

Argentina

In [126]:
preview_manual_xi("Argentina", [
    "Emiliano Martinez",   # GK
    "Nahuel Molina",       # RB
    "Cristian Romero",       # CB
    "Nicolas Otamendi",        # CB
    "Nicolas Tagliafico",        # LB
    "Enzo Fernandez",   # CM
    "Alexis Mac Allister",        # CM
    "Rodrigo de Paul",       # RW
    "Lautaro Martinez",
    "Lionel Messi",        # ST
    "Julian Alvarez",    # LW
])

=== Argentina preview (55 squad players, total xMins 990.0) ===

               player position is_starter  mp_share  conditional_min  xmins
    Emiliano Martínez       GK       True      0.90            87.50  87.95
      Cristian Romero      DEF       True      0.83            83.33  83.78
       Enzo Fernández      MID       True      0.83            80.78  81.23
      Rodrigo De Paul      MID       True      0.93            80.33  80.78
   Nicolás Tagliafico      DEF       True      0.86            80.20  80.64
         Lionel Messi      FWD       True      0.69            78.54  78.99
     Nicolás Otamendi      DEF       True      0.93            77.59  78.04
        Nahuel Molina      DEF       True      0.88            76.55  77.00
  Alexis Mac Allister      MID       True      0.79            76.16  76.61
       Julián Alvarez      FWD       True      0.98            73.45  73.90
     Lautaro Martínez      FWD       True      0.81            55.68  56.13
      Leandro Paredes  

### Group K

Portugal

In [127]:
preview_manual_xi("Portugal", [
    "Diogo Costa",   # GK
    "Joao Cancelo",       # RB
    "Ruben Dias",       # CB
    "Goncalo Inacio",        # CB
    "Nuno Mendes",        # LB
    "Vitinha",       # CDM
    "Joao Neves",   # CM
    "Bruno Fernandes",        # CM
    "Bernardo Silva",       # RW
    "Rafael Leao",        # ST
    "Cristiano Ronaldo",    # LW
])

=== Portugal preview (27 squad players, total xMins 990.0) ===

             player position is_starter  mp_share  conditional_min  xmins
        Diogo Costa       GK       True      0.95            90.00  90.00
         Rúben Dias      DEF       True      0.88            89.05  89.05
        Nuno Mendes      DEF       True      0.84            87.39  87.39
    Bruno Fernandes      MID       True      0.86            85.59  85.59
            Vitinha      MID       True      0.88            80.26  80.26
  Cristiano Ronaldo      FWD       True      0.89            79.55  79.55
     Bernardo Silva      MID       True      0.86            77.44  77.44
       João Cancelo      DEF       True      0.47            77.22  77.22
     Gonçalo Inácio      DEF       True      0.48            70.16  70.16
         João Neves      MID       True      0.64            68.26  68.26
        Rafael Leão      MID       True      0.73            61.93  61.93
         Pedro Neto      MID      False      0.6

Colombia

In [130]:
preview_manual_xi("Colombia", [
    "Camilo Vargas",   # GK
    "Daniel Munoz",       # RB
    "Davinson Sanchez",       # CB
    "Jhon Lucumi",        # CB
    "Johan Mojica",        # LB
    "Jefferson Lerma",       # CDM
    "Richard Rios",   # CM
    "Jhon Arias",        # CM
    "James Rodriguez",       # RW
    "Luis Diaz",        # ST
    "Jhon Cordoba",    # LW
])

=== Colombia preview (26 squad players, total xMins 990.0) ===

                player position is_starter  mp_share  conditional_min  xmins
         Camilo Vargas       GK       True      0.81            87.09  87.80
      Dávinson Sánchez      DEF       True      0.71            85.10  85.81
          Daniel Muñoz      DEF       True      0.74            84.78  85.49
           Jhon Lucumí      DEF       True      0.69            84.49  85.20
          Johan Mojica      DEF       True      0.52            84.47  85.18
             Luis Díaz      MID       True      0.95            83.74  84.45
       Jefferson Lerma      MID       True      0.79            77.77  78.48
       James Rodríguez      MID       True      1.00            70.98  71.69
            Jhon Arias      MID       True      0.95            70.22  70.93
          Richard Ríos      MID       True      0.86            68.91  69.62
          Jhon Córdoba      FWD       True      0.62            60.67  61.38
         Kev

### Group L

England

In [ ]:
preview_manual_xi("England", [
    "Jordan Pickford",   # GK
    "Reece James",       # RB
    "John Stones",       # CB
    "Marc Guehi",        # CB
    "Nico O'Reilly",        # LB
    "Declan Rice",       # CDM
    "Elliot Anderson",   # CM
    "Jude Bellingham",        # CM
    "Bukayo Saka",       # RW
    "Harry Kane",        # ST
    "Marcus Rashford",    # LW
])

=== England preview (26 squad players, total xMins 990.0) ===

          player position is_starter  mp_share  conditional_min  xmins
 Jordan Pickford       GK       True      0.93            87.66  87.93
      Marc Guéhi      DEF       True      0.64            80.90  81.17
     John Stones      DEF       True      0.62            80.60  80.87
 Jude Bellingham      MID       True      0.75            80.35  80.62
      Harry Kane      FWD       True      1.00            79.67  79.94
   Nico O'Reilly      DEF       True      0.13            79.29  79.56
     Declan Rice      MID       True      0.95            76.28  76.55
 Elliot Anderson      MID       True      0.33            74.50  74.77
     Bukayo Saka      MID       True      0.57            74.38  74.65
     Reece James      DEF       True      0.39            71.00  71.27
 Marcus Rashford      MID       True      0.46            59.58  59.85
      Ezri Konsa      DEF      False      0.66            70.35  26.99
   Morgan Roge

Croatia

In [132]:
preview_manual_xi("Croatia", [
    "Dominik Livakovic",   # GK
    "Josip Stanisic",       # RB
    "Josip Sutalo",       # CB
    "Duje Caleta-Car",        # CB
    "Josko Gvardiol",        # LB
    "Luka Modric",       # CDM
    "Mateo Kovacic",   # CM
    "Mario Pasalic",        # CM
    "Andrej Kramaric",       # RW
    "Ante Budimir",        # ST
    "Ivan Perisic",    # LW
])

=== Croatia preview (26 squad players, total xMins 990.0) ===

           player position is_starter  mp_share  conditional_min  xmins
Dominik Livakovic       GK       True      0.89            86.60  86.60
     Josip Sutalo      DEF       True      0.81            84.43  84.43
  Duje Caleta-Car      DEF       True      0.73            81.68  81.68
   Josko Gvardiol      DEF       True      0.81            80.37  80.37
   Josip Stanisic      DEF       True      0.48            80.12  80.12
    Mateo Kovacic      MID       True      0.45            75.83  75.83
  Andrej Kramaric      FWD       True      1.00            70.12  70.12
      Luka Modric      MID       True      1.00            68.46  68.46
     Ivan Perisic      FWD       True      1.00            63.87  63.87
    Mario Pasalic      MID       True      0.94            58.94  58.94
     Ante Budimir      FWD       True      0.71            53.72  53.72
      Petar Sucic      MID      False      0.73            68.95  33.79
 